### **Import Library**

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.naive_bayes import MultinomialNB, CategoricalNB, GaussianNB
from sklearn.metrics import accuracy_score
from category_encoders import CatBoostEncoder

### Read dataset

In [2]:
df = pd.read_csv("../Data/diabetes_70k.csv")

# Separate features and target
X = df.drop(columns=["class"])
y = df["class"]

print("Number of rows:", X.shape[0])
print("Number of features:", X.shape[1])
print("Class distribution:")
print(y.value_counts())

Number of rows: 70692
Number of features: 21
Class distribution:
class
0    35346
1    35346
Name: count, dtype: int64


**Task 1 — OneHotEncoder vs OrdinalEncoder with Naive Bayes**  
Using scikit-learn, evaluate the following encoder–classifier combinations:  
• OneHotEncoder → MultinomialNB  
• OrdinalEncoder → CategoricalNB  
For this task:
1. Apply each encoder to the dataset
2. Use a 50:50 train–test split.
3. Train the classifier.
4. Repeat the experiment multiple times (e.g., 20 runs) using different random seeds.
5. Report the average accuracy for each encoder.

##### OneHotEncoder with MultinomialNB

In [3]:
def run_onehot_multinomial(X_train, X_test, y_train, y_test):
    # Categories unseen in the training set are all 0
    ohe = OneHotEncoder(handle_unknown="ignore")
    X_train_enc = ohe.fit_transform(X_train)
    X_test_enc = ohe.transform(X_test)

    model = MultinomialNB()
    model.fit(X_train_enc, y_train)

    return accuracy_score(y_test, model.predict(X_test_enc))

##### OrdinalEncoder with CategoricalNB

In [4]:
def run_ordinal_categorical(X_train, X_test, y_train, y_test):
    # Unseen categories in the test set are first mark as -1
    ordinal = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1
    )
    X_train_enc = ordinal.fit_transform(X_train)
    X_test_enc = ordinal.transform(X_test)

    # Number of categories per column
    n_cats = np.array([len(c) for c in ordinal.categories_])

    # Move unseen value to extra index
    X_test_enc = np.where(X_test_enc == -1, n_cats, X_test_enc).astype(int)
    X_train_enc = X_train_enc.astype(int)

    model = CategoricalNB(min_categories=n_cats + 1)
    model.fit(X_train_enc, y_train)

    return accuracy_score(y_test, model.predict(X_test_enc))

##### Repeat the experiment 20 times with 50:50

In [5]:
n_runs = 20
ohe_acc = []
ord_acc = []

for seed in range(n_runs):
    # 50:50 split with a different rand seed
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.5, random_state=seed
    )

    acc_ohe = run_onehot_multinomial(X_train, X_test, y_train, y_test)
    acc_ord = run_ordinal_categorical(X_train, X_test, y_train, y_test)

    ohe_acc.append(acc_ohe)
    ord_acc.append(acc_ord)

    print(f"Run {seed + 1}")
    print(f"OneHotEncoder + MultinomialNB accuracy: {acc_ohe:.4f}")
    print(f"OrdinalEncoder + CategoricalNB accuracy: {acc_ord:.4f}")
    print()

Run 1
OneHotEncoder + MultinomialNB accuracy: 0.7365
OrdinalEncoder + CategoricalNB accuracy: 0.7365

Run 2
OneHotEncoder + MultinomialNB accuracy: 0.7342
OrdinalEncoder + CategoricalNB accuracy: 0.7342

Run 3
OneHotEncoder + MultinomialNB accuracy: 0.7370
OrdinalEncoder + CategoricalNB accuracy: 0.7370

Run 4
OneHotEncoder + MultinomialNB accuracy: 0.7297
OrdinalEncoder + CategoricalNB accuracy: 0.7297

Run 5
OneHotEncoder + MultinomialNB accuracy: 0.7334
OrdinalEncoder + CategoricalNB accuracy: 0.7334

Run 6
OneHotEncoder + MultinomialNB accuracy: 0.7331
OrdinalEncoder + CategoricalNB accuracy: 0.7331

Run 7
OneHotEncoder + MultinomialNB accuracy: 0.7345
OrdinalEncoder + CategoricalNB accuracy: 0.7345

Run 8
OneHotEncoder + MultinomialNB accuracy: 0.7339
OrdinalEncoder + CategoricalNB accuracy: 0.7339

Run 9
OneHotEncoder + MultinomialNB accuracy: 0.7330
OrdinalEncoder + CategoricalNB accuracy: 0.7330

Run 10
OneHotEncoder + MultinomialNB accuracy: 0.7324
OrdinalEncoder + Categorical

##### The avg accuracy

In [6]:
print("Average accuracy over", n_runs, "runs")
print(f"OneHotEncoder + MultinomialNB: {np.mean(ohe_acc):.4f}")
print(f"OrdinalEncoder + CategoricalNB: {np.mean(ord_acc):.4f}")

Average accuracy over 20 runs
OneHotEncoder + MultinomialNB: 0.7333
OrdinalEncoder + CategoricalNB: 0.7333


**Task 2 — CatBoostEncoder with GaussianNB**  
Use the CatBoostEncoder from the category_encoders package:  
pip install category_encoders  
Because CatBoostEncoder produces continuous-valued features, use:  
• CatBoostEncoder → GaussianNB  
Repeat the same evaluation procedure as in Task 1.

##### CatBoostEncoder with GaussianNB

In [7]:
def run_catboost_gaussian(X_train, X_test, y_train, y_test):

    # Encode every cols as it skip num col by default
    encoder = CatBoostEncoder(cols=list(X_train.columns))
    X_train_enc = encoder.fit_transform(X_train, y_train)

    X_test_enc = encoder.transform(X_test)

    model = GaussianNB()
    model.fit(X_train_enc, y_train)

    return accuracy_score(y_test, model.predict(X_test_enc))

##### Repeat the experiment 20 times with 50:50

In [8]:
n_runs = 20
cat_acc = []

for seed in range(n_runs):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.5, random_state=seed
    )

    acc_cat = run_catboost_gaussian(X_train, X_test, y_train, y_test)
    cat_acc.append(acc_cat)

    print(f"Run {seed + 1}")
    print(f"CatBoostEncoder + GaussianNB accuracy: {acc_cat:.4f}")
    print()

Run 1
CatBoostEncoder + GaussianNB accuracy: 0.7318

Run 2
CatBoostEncoder + GaussianNB accuracy: 0.7286

Run 3
CatBoostEncoder + GaussianNB accuracy: 0.7329

Run 4
CatBoostEncoder + GaussianNB accuracy: 0.7246

Run 5
CatBoostEncoder + GaussianNB accuracy: 0.7288

Run 6
CatBoostEncoder + GaussianNB accuracy: 0.7279

Run 7
CatBoostEncoder + GaussianNB accuracy: 0.7291

Run 8
CatBoostEncoder + GaussianNB accuracy: 0.7261

Run 9
CatBoostEncoder + GaussianNB accuracy: 0.7281

Run 10
CatBoostEncoder + GaussianNB accuracy: 0.7265

Run 11
CatBoostEncoder + GaussianNB accuracy: 0.7249

Run 12
CatBoostEncoder + GaussianNB accuracy: 0.7270

Run 13
CatBoostEncoder + GaussianNB accuracy: 0.7332

Run 14
CatBoostEncoder + GaussianNB accuracy: 0.7307

Run 15
CatBoostEncoder + GaussianNB accuracy: 0.7292

Run 16
CatBoostEncoder + GaussianNB accuracy: 0.7276

Run 17
CatBoostEncoder + GaussianNB accuracy: 0.7255

Run 18
CatBoostEncoder + GaussianNB accuracy: 0.7251

Run 19
CatBoostEncoder + GaussianNB a

##### The avg accuracy

In [9]:
print("Average accuracy over", n_runs, "runs")
print(f"CatBoostEncoder + GaussianNB: {np.mean(cat_acc):.4f}")

Average accuracy over 20 runs
CatBoostEncoder + GaussianNB: 0.7282


**Task 3 — Fit Encoders After Splitting the Data**  
Repeat Tasks 1 and 2, but change the workflow:  
1. Split the dataset into training and testing sets first.
2. Fit each encoder only on the training data.
3. Transform both training and test sets using the fitted encoder.
4. Evaluate the same three encoder–classifier combinations.
Explain data leakage and how it relates to this task.


##### Encode before split

In [10]:
def leaky_onehot_multinomial(X, y, seed):
    X_enc = OneHotEncoder().fit_transform(X)
    X_train, X_test, y_train, y_test = train_test_split(
        X_enc, y, test_size=0.5, random_state=seed
    )
    model = MultinomialNB().fit(X_train, y_train)
    return accuracy_score(y_test, model.predict(X_test))


def leaky_ordinal_categorical(X, y, seed):
    X_enc = OrdinalEncoder().fit_transform(X).astype(int)
    X_train, X_test, y_train, y_test = train_test_split(
        X_enc, y, test_size=0.5, random_state=seed
    )
    model = CategoricalNB(min_categories=X_enc.max(axis=0) + 1)
    model.fit(X_train, y_train)
    return accuracy_score(y_test, model.predict(X_test))


def leaky_catboost_gaussian(X, y, seed):
    X_enc = CatBoostEncoder(cols=list(X.columns)).fit_transform(X, y)
    X_train, X_test, y_train, y_test = train_test_split(
        X_enc, y, test_size=0.5, random_state=seed
    )
    model = GaussianNB().fit(X_train, y_train)
    return accuracy_score(y_test, model.predict(X_test))

##### Run the encode before split 20 times

In [11]:
leaky_ohe, leaky_ord, leaky_cat = [], [], []

for seed in range(n_runs):
    leaky_ohe.append(leaky_onehot_multinomial(X, y, seed))
    leaky_ord.append(leaky_ordinal_categorical(X, y, seed))
    leaky_cat.append(leaky_catboost_gaussian(X, y, seed))

print("Average accuracy, encoder fitted before the split")
print(f"OneHotEncoder + MultinomialNB: {np.mean(leaky_ohe):.4f}")
print(f"OrdinalEncoder + CategoricalNB: {np.mean(leaky_ord):.4f}")
print(f"CatBoostEncoder + GaussianNB: {np.mean(leaky_cat):.4f}")

Average accuracy, encoder fitted before the split
OneHotEncoder + MultinomialNB: 0.7333
OrdinalEncoder + CategoricalNB: 0.7333
CatBoostEncoder + GaussianNB: 0.9901


##### Task 3 workflow

In [12]:
print("Fitted before the split versus after the split")
print()
for name, before, after in [
    ("OneHotEncoder + MultinomialNB", leaky_ohe, ohe_acc),
    ("OrdinalEncoder + CategoricalNB", leaky_ord, ord_acc),
    ("CatBoostEncoder + GaussianNB", leaky_cat, cat_acc),
]:
    print(name)
    print(f"Before split: {np.mean(before):.4f}")
    print(f"After split: {np.mean(after):.4f}")
    print(f"Difference: {np.mean(before) - np.mean(after):.4f}")
    print()

Fitted before the split versus after the split

OneHotEncoder + MultinomialNB
Before split: 0.7333
After split: 0.7333
Difference: 0.0000

OrdinalEncoder + CategoricalNB
Before split: 0.7333
After split: 0.7333
Difference: 0.0000

CatBoostEncoder + GaussianNB
Before split: 0.9901
After split: 0.7282
Difference: 0.2619



##### Only CatBoostEncoder shows a visible gap, because it is the only encoder that uses the target value. Fitting it before the split lets the test labels influence the encoded features, which affects the test dataset.

##### **Data Leakage**  
##### Data leakage happens when information from outside the training data, usually from the test set, influences the model or its preprocessing. The test result then looks better than the model would perform on truly new data. The test set is supposed to act like future data the model has never seen, so anything learned from it during training breaks that assumption.  

##### **How its relate**  
##### CatBoostEncoder calculates its encoded values from the target. If we encode before the split, the test labels are used in that calculation, so the test data is affected and the accuracy becomes too optimistic.